<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/IBM_Granite4_1_30b_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploying IBM Granite 4.1:30b in Google Colaboratory with `uv` and Ollama

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)


## Primer

This notebook deploys the IBM Granite 4.1 30B language model in a Google Colaboratory GPU runtime using `uv` for Python package management and Ollama for local inference serving. The configuration is tuned for a high memory G4 class Colab runtime with roughly 96 GB of GPU memory, although the cells include runtime checks that report the actual device and visible memory before the model is pulled.

Granite 4.1 is IBM's dense language model family in 3B, 8B, and 30B sizes. The 30B instruction model is appropriate for longer technical prompts, code generation, retrieval augmented generation, tool use experiments, and structured JSON output. Ollama exposes the requested runtime model as the lowercase tag `granite4.1:30b`, and this notebook preserves the human readable model label as `Granite 4.1:30b`.

| Term | Meaning |
|---|---|
| Colab | Google Colaboratory, a cloud hosted Jupyter environment. |
| GPU | Graphics Processing Unit, the accelerator used for model inference. |
| VRAM | GPU memory used for model weights, key value cache, and runtime buffers. |
| Ollama | A local model server and client interface for running open models. |
| `uv` | A fast Python package installer and resolver. |
| Context window | The maximum prompt plus generation token budget accepted by the model runtime. |

The notebook follows a linear execution path. It first verifies the runtime hardware, then installs system dependencies, installs `uv`, installs Ollama, starts `ollama serve`, pulls `granite4.1:30b`, installs the Python client with `uv pip`, runs a smoke test, and exposes a persistent chat loop.

Reference notes used for this notebook were checked on July 8, 2026. IBM lists Granite 4.1 as a dense 3B, 8B, and 30B model family, and Ollama lists `granite4.1:30b` with a 128K context window under the Granite 4.1 model tags.


## Control knobs

Adjust these values before running the installation and inference cells.

| Name | Meaning |
|---|---|
| `llm_version` | Human readable model label retained as `Granite 4.1:30b`. |
| `model_name` | Ollama model tag used by pull and chat calls. |
| `ollama_host` | Host interface for the local Ollama server. |
| `ollama_port` | TCP port used by the local Ollama server. |
| `models_dir` | Directory for downloaded Ollama model files. |
| `log_path` | Log file for `ollama serve`. |
| `cuda_visible_devices` | GPU device selector exposed to Ollama. |
| `num_ctx` | Requested context window size for chat calls. |
| `num_predict` | Maximum generated tokens per response. |
| `temperature` | Sampling temperature used for chat calls. |
| `top_p` | Nucleus sampling threshold used for chat calls. |
| `uv_bin_dir` | Install location for the `uv` binary. |
| `install_uv` | Whether to install `uv`. |
| `install_ollama` | Whether to install Ollama. |


In [ ]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import time
import urllib.error
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional


@dataclass(frozen=True)
class ColabOllamaConfig:
    """Configuration for running Granite 4.1:30b in Colab."""

    llm_version: str = "Granite 4.1:30b"
    model_name: str = "granite4.1:30b"
    ollama_host: str = "127.0.0.1"
    ollama_port: int = 11434
    models_dir: Path = Path("/content/ollama_models")
    log_path: Path = Path("/content/ollama_serve.log")
    cuda_visible_devices: str = "0"
    num_ctx: int = 131072
    num_predict: int = 512
    temperature: float = 0.20
    top_p: float = 0.90
    uv_bin_dir: Path = Path("/content/.local/bin")
    install_uv: bool = True
    install_ollama: bool = True


CFG = ColabOllamaConfig()
print(json.dumps(asdict(CFG), indent=2, default=str))


## Runtime helper functions

These functions keep the remaining cells compact and provide deterministic checks around system commands, GPU visibility, TCP server readiness, and model capacity warnings.


In [ ]:
def _is_root() -> bool:
    """Return True if the current process has root privileges."""
    try:
        return os.geteuid() == 0
    except AttributeError:
        return False


def _sudo_prefix() -> str:
    """Return a sudo prefix when root privileges are absent."""
    return "" if _is_root() else "sudo "


def run_bash(command: str, *, check: bool = True) -> subprocess.CompletedProcess:
    """Run a bash command in a Colab friendly way."""
    print(f"\n[run] {command}\n")
    return subprocess.run(["bash", "-lc", command], check=check)


def capture_bash(command: str) -> str:
    """Run a bash command and capture stdout as text."""
    out = subprocess.check_output(["bash", "-lc", command], text=True)
    return out.strip()


def prepend_to_path(path: Path) -> None:
    """Prepend a directory to PATH for later subprocess calls."""
    path_str = str(path)
    path_parts = os.environ.get("PATH", "").split(":")
    if path_str not in path_parts:
        os.environ["PATH"] = f"{path_str}:{os.environ.get('PATH', '')}"


def is_tcp_port_open(host: str, port: int, timeout_s: float = 0.25) -> bool:
    """Return True if a TCP port accepts connections."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout_s)
        return sock.connect_ex((host, port)) == 0


def wait_for_ollama_ready(
    host: str,
    port: int,
    timeout_s: float = 45.0,
    poll_s: float = 0.5,
) -> dict[str, Any]:
    """Wait until the Ollama server responds to GET /api/version."""
    url = f"http://{host}:{port}/api/version"
    deadline = time.time() + timeout_s
    last_err: Optional[Exception] = None

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2.0) as resp:
                payload = resp.read().decode("utf-8")
            return json.loads(payload)
        except (urllib.error.URLError, json.JSONDecodeError) as err:
            last_err = err
            time.sleep(poll_s)

    raise TimeoutError(f"Ollama did not become ready: {last_err}")


def nvidia_smi_summary() -> str:
    """Return a concise GPU summary from nvidia-smi."""
    command = "command -v nvidia-smi >/dev/null 2>&1"
    if subprocess.call(["bash", "-lc", command]) != 0:
        return "nvidia-smi not found. Select a Colab GPU runtime."

    query = (
        "nvidia-smi --query-gpu=index,name,driver_version,"
        "memory.total,memory.free --format=csv,noheader,nounits"
    )
    return capture_bash(query)


def parse_gpu_memory_gib(gpu_summary: str) -> list[float]:
    """Parse total GPU memory values in GiB from nvidia-smi CSV output."""
    memory_gib: list[float] = []

    for line in gpu_summary.splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 4:
            continue

        try:
            memory_gib.append(float(parts[3]) / 1024.0)
        except ValueError:
            continue

    return memory_gib


def warn_for_granite4_1_30b_capacity(
    gpu_summary: str,
    target_gib: float = 90.0,
) -> None:
    """Print a Granite 4.1 30B capacity note from visible GPU memory."""
    memory_gib = parse_gpu_memory_gib(gpu_summary)
    upper = gpu_summary.upper()

    if not memory_gib:
        print(
            "[gpu] WARNING: GPU memory could not be parsed. Confirm that the "
            "runtime is attached to a high memory GPU before pulling the model."
        )
        return

    max_memory = max(memory_gib)
    print(f"[gpu] Largest visible GPU memory: {max_memory:.1f} GiB")

    if "G4" in upper or max_memory >= target_gib:
        print(
            "[gpu] High memory runtime detected. This is the intended target "
            f"for {CFG.llm_version} with a large context window."
        )
        return

    if max_memory >= 40.0:
        print(
            "[gpu] The model can often load with quantized weights on this class "
            "of device, although a smaller `CFG.num_ctx` may be required."
        )
        return

    print(
        "[gpu] WARNING: This runtime is likely undersized for comfortable "
        f"{CFG.llm_version} inference. Reduce `CFG.num_ctx` or use a larger GPU."
    )


def ollama_base_url() -> str:
    """Return the configured Ollama HTTP base URL."""
    return f"http://{CFG.ollama_host}:{CFG.ollama_port}"


def chat_options() -> dict[str, int | float]:
    """Return shared Ollama chat generation options."""
    return {
        "num_ctx": CFG.num_ctx,
        "num_predict": CFG.num_predict,
        "temperature": CFG.temperature,
        "top_p": CFG.top_p,
    }


## Runtime sanity checks

Run this cell first. It confirms GPU visibility, reports GPU memory, and checks available disk space under `/content`, which is where the model directory is located by default.


In [ ]:
gpu_summary = nvidia_smi_summary()
print(gpu_summary)
warn_for_granite4_1_30b_capacity(gpu_summary)

run_bash("df -h /content")
run_bash("uname -a")


## Install baseline packages, `uv`, and Ollama

This cell installs Linux utilities, installs `uv` in unmanaged mode, installs Ollama using the official Linux installer, and verifies that both command line tools are available.


In [ ]:
sudo = _sudo_prefix()

if CFG.install_ollama or CFG.install_uv:
    run_bash(f"{sudo}apt-get update -y")
    run_bash(
        f"{sudo}apt-get install -y "
        "curl ca-certificates zstd pciutils lshw"
    )

if CFG.install_uv:
    CFG.uv_bin_dir.mkdir(parents=True, exist_ok=True)
    run_bash(
        "curl -LsSf https://astral.sh/uv/install.sh | "
        f'env UV_UNMANAGED_INSTALL="{CFG.uv_bin_dir}" sh'
    )

prepend_to_path(CFG.uv_bin_dir)
run_bash("command -v uv")
run_bash("uv --version")

if CFG.install_ollama:
    run_bash("curl -fsSL https://ollama.com/install.sh | sh")

run_bash("command -v ollama")
run_bash("ollama -v")


## Start `ollama serve` and pull `granite4.1:30b`

This cell starts the local Ollama server, waits for the HTTP version endpoint, pulls the requested model, and prints the local model inventory. The server log is written to `CFG.log_path`.


In [ ]:
CFG.models_dir.mkdir(parents=True, exist_ok=True)

if is_tcp_port_open(CFG.ollama_host, CFG.ollama_port):
    print(
        "[ollama] Server already listening on "
        f"{CFG.ollama_host}:{CFG.ollama_port}"
    )
else:
    env = os.environ.copy()
    env["OLLAMA_HOST"] = f"{CFG.ollama_host}:{CFG.ollama_port}"
    env["OLLAMA_MODELS"] = str(CFG.models_dir)
    env["CUDA_VISIBLE_DEVICES"] = CFG.cuda_visible_devices

    CFG.log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ollama] Logging to: {CFG.log_path}")

    with CFG.log_path.open("a", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            ["ollama", "serve"],
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )

    print(f"[ollama] Started server PID={proc.pid}")

version_payload = wait_for_ollama_ready(CFG.ollama_host, CFG.ollama_port)
print("[ollama] /api/version =>", version_payload)

run_bash(f"ollama pull {CFG.model_name}")
run_bash("ollama list")
run_bash(f"ollama show {CFG.model_name}", check=False)


## Install the Ollama Python client with `uv`

This cell installs the Python client into the active Colab environment so the remaining cells can call the local server from Python.


In [ ]:
run_bash("uv pip install --system --upgrade ollama")


## Smoke test

The smoke test sends a short technical prompt to `Granite 4.1:30b`, prints the response, then reports active Ollama model placement and GPU memory utilization.


In [ ]:
from ollama import chat

smoke_messages = [
    {
        "role": "system",
        "content": (
            "You are a precise technical assistant. Reply in one compact "
            "paragraph unless the user requests a longer structure."
        ),
    },
    {
        "role": "user",
        "content": (
            "Confirm that you are running as Granite 4.1:30b and explain, "
            "in one paragraph, what this local Colab deployment is doing."
        ),
    },
]

response = chat(
    model=CFG.model_name,
    messages=smoke_messages,
    options=chat_options(),
)

print(response.message.content)
run_bash("ollama ps")
run_bash("nvidia-smi")


## Streaming response test

This cell verifies token streaming, which is useful for longer technical prompts where an immediate partial response is preferable.


In [ ]:
from ollama import chat

stream_messages = [
    {
        "role": "user",
        "content": (
            "Give a concise three point checklist for validating a local "
            "Granite 4.1:30b deployment in Colab."
        ),
    }
]

for chunk in chat(
    model=CFG.model_name,
    messages=stream_messages,
    options=chat_options(),
    stream=True,
):
    print(chunk.message.content, end="", flush=True)

print()


## Interactive chat loop

Run this cell after the model is installed. Type `quit`, `exit`, or `q` to stop the loop. Reduce `CFG.num_ctx` in the configuration cell if memory pressure appears during long conversations.


In [ ]:
from ollama import chat

MODEL_NAME = CFG.model_name
NUM_CTX = CFG.num_ctx
SYSTEM_PROMPT = (
    "You are Granite 4.1:30b running locally in Google Colab. "
    "You provide concise, technically precise answers."
)

messages: list[dict[str, str]] = []
if SYSTEM_PROMPT.strip():
    messages.append({"role": "system", "content": SYSTEM_PROMPT.strip()})


def send_turn(user_text: str) -> str:
    """Send one user turn and return one assistant turn."""
    messages.append({"role": "user", "content": user_text})
    response = chat(
        model=MODEL_NAME,
        messages=messages,
        options=chat_options(),
    )
    assistant_text = response.message.content
    messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text


while True:
    user_text = input("\nYou: ").strip()
    if user_text.lower() in {"quit", "exit", "q"}:
        print("Stopped.")
        break
    if not user_text:
        continue

    try:
        reply = send_turn(user_text)
    except Exception as exc:
        raise RuntimeError(
            "Prompting failed. Verify these earlier cells:\n"
            "  1) `ollama serve` is running on localhost:11434\n"
            f"  2) the model is present with `ollama pull {CFG.model_name}`\n"
            "  3) the `ollama` Python package is installed\n"
            "  4) `CFG.num_ctx` fits the visible GPU memory\n"
        ) from exc

    print(f"\nAssistant: {reply}")


## Save a transcript

Run this optional cell after the interactive loop to save the conversation into `/content/granite_4_1_30b_chat_transcript.json`.


In [ ]:
transcript_path = Path("/content/granite_4_1_30b_chat_transcript.json")

if "messages" not in globals() or not messages:
    print("No chat transcript is available yet.")
else:
    transcript_path.write_text(
        json.dumps(messages, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Saved transcript to {transcript_path}")


## Operational notes

Reconnecting to a new Colab runtime means the local server, packages, and downloaded model may need to be recreated. The default model directory is `/content/ollama_models`, so the model files disappear when the runtime is recycled unless they are copied to persistent storage. The default model tag in this notebook is `granite4.1:30b`, matching the requested `Granite 4.1:30b` version label. Smaller context windows such as 32768 or 65536 tokens can be used if a runtime reports less memory than expected.
